In [3]:
from pathlib import Path

MODEL_ID = "google/gemma-4-E2B-it"
WEIGHTS  = Path("model_weights")

In [4]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Run our PyTorch-side tokenizer
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from architecture.tokenization import GemmaTokenizer

tok = GemmaTokenizer(WEIGHTS / "tokenizer.json")

s = "Explain MoE in transformers in 3 sentences."
ids = tok.encode(s)

print(f"vocab_size = {tok.vocab_size:,}")
print(f"ids ({len(ids)}): {ids[:10]} ...")

for tid, piece in tok.pretty(ids)[:10]:
    print(f"  {tid:>6}  {piece!r}")

print(f"\nroundtrip: {tok.decode(ids)!r}")

vocab_size = 262,144
ids (10): [155122, 7945, 236788, 528, 39687, 528, 236743, 236800, 23974, 236761] ...
  155122  'Explain'
    7945  '▁Mo'
  236788  'E'
     528  '▁in'
   39687  '▁transformers'
     528  '▁in'
  236743  '▁'
  236800  '3'
   23974  '▁sentences'
  236761  '.'

roundtrip: 'Explain MoE in transformers in 3 sentences.'


In [6]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Correction: E2B ships a single model.safetensors (no index / no shards)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from huggingface_hub import hf_hub_download

shard_path = hf_hub_download(MODEL_ID, "model.safetensors", local_dir=WEIGHTS)
print(f"downloaded → {shard_path}")

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

downloaded → model_weights/model.safetensors


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Embed the token ids → residual stream vectors
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import torch
from architecture.embedding import GemmaEmbedding

emb = GemmaEmbedding.from_safetensors(shard_path)
print(f"embedding: {tuple(emb.embed.weight.shape)}  dtype={emb.embed.weight.dtype}")

ids_t = torch.tensor(ids, dtype=torch.long)
vecs  = emb(ids_t)

print(f"ids   → {tuple(ids_t.shape)}")
print(f"vecs  → {tuple(vecs.shape)}")
print(f"vecs[0, :8] = {vecs[0, :8].tolist()}")